# MaxQuant → Percolator Feature Conversion

This notebook converts **MaxQuant `msms.txt` output** into a
Percolator-compatible feature file (`.pin`-style tab-delimited format).

Feature definitions follow the Crux/Percolator specification:  
https://crux.ms/file-formats/features.html   https://crux.ms/file-formats/features.html [2025-08-14]

# Input Files

#### `msms.txt`
MaxQuant PSM-level output containing:
- Scan number
- Raw file
- m/z (precursor)
- Mass
- Score
- Delta score
- Charge
- Peak coverage
- Modified sequence
- Proteins
- Evidence ID
- Fragment match information
-  missed cleavages

#### `peptides.txt`
Used to:
- Reconstruct enzymatic digestion features

---

# Converting MaxQuant Output to Percolator PIN Format

The following Percolator-required columns are generated:

| Percolator Column | MaxQuant Source | Notes |
|-------------------|-----------------|-------|
| **SpecId** | Constructed | Format: `target_0_scan_rawfile_charge` |
| **Label** | `Reverse` | target = 1, decoy = -1 |
| **ScanNr** | `Scan number` | |
| **ExpMass** | `m/z` | Precursor m/z |
| **CalcMass** | `Mass` | Calculated peptide mass |
| **lnrSp** | Rank of Score | log(rank) |
| **deltLCn** | Score vs last score |  |
| **deltCn** | Delta score |  |
| **Sp** | `Score` | Main MQ score |
| **IonFrac** | `Peak coverage` | |
| **Mass** | `Mass` | |
| **PepLen** | `Length` | Peptide length |
| **ChargeN** | `Charge` | Integer charge |
| **enzInt** | `Missed cleavages` | |
| **enzN** | Derived | N-terminal enzymatic cleavage flag |
| **enzC** | Derived | C-terminal enzymatic cleavage flag |
| **Peptide** | `Modified sequence` | Reformatted to bracket notation |
| **Proteins** | `Proteins` | Semicolon → comma separated |

---

# Endogenous SUMO Data – Digestion Rules

SUMO workflows follow custom digestion logic.

### Enzymes Used

- **Asp-N**  
  Cleaves N-terminal to Asp (D)

- **Lys-C**  
  Cleaves C-terminal to Lys (K)

---

## SUMO-Specific Enzymatic Rules

### N-Terminal (`enzN`)

Peptide is considered enzymatic if:
- Previous amino acid is K  
OR  
- Peptide starts with D or E  
OR  
- No previous residue (protein N-terminus)

---

### C-Terminal (`enzC`)

Peptide is considered enzymatic if:
- Ends with K  
OR  
- Next amino acid is D or E  
OR  
- Protein C-terminus  

Special SUMO rule:
- If SUMO is on terminal K, the following residue must be D or E.

---


# Workflow Summary

1. Read `msms.txt`
2. Expand and merge with `peptides.txt`
3. Engineer Percolator features
4. Apply modification-specific digestion rules
5. Reformat sequences
6. Export tab-delimited Percolator input file

---

# Output

A Percolator-compatible feature file


In [1]:
import pandas as pd
import os
import numpy as np

In [6]:
input_file = 'msms.txt'
peptides_table_sbm3 = 'peptides.txt'
output_file_sbm3  = 'percolator_pin'

In [4]:
def convert_maxquant_to_percolator(input_file, output_file, peptides_table, mod):

     """
    Convert MaxQuant msms output into Percolator-compatible input format.

    Merges MaxQuant evidence with the peptides table to reconstruct digestion
    context, computes Percolator feature columns, and reformats modified
    peptide sequences depending on modification type (SUMO, ubiquitin, or none).

    Parameters
    ----------
    input_file : str
        Path to MaxQuant msms.txt (or similar) file.
    output_file : str
        Path to save the generated Percolator input (.txt).
    peptides_table : str
        Path to MaxQuant peptides.txt file (used for digestion features).
    mod : str
        Modification type: 'sumo' or 'none'.

    Returns
    -------
    pandas.DataFrame
        Percolator-formatted feature table.

    Notes
    -----
    - Generates standard Percolator fields such as:
      SpecId, Label, ScanNr, ExpMass, CalcMass, lnrSp,
      deltLCn, deltCn, Sp, IonFrac, enzN, enzC, etc.
    - Reformats modified sequences to bracket notation (e.g. K[su]).
    - Digestion rules are customized for SUMO and ubiquitin workflows.
    """
    
    # Load MaxQuant data msms.txt
    df = pd.read_csv(input_file, sep='\t') 
    
    # read peptides table for extracting the digestion pattern
    df_peps = pd.read_csv(peptides_table, delimiter="\t")
    df_peps['Evidence ID'] = df_peps['Evidence IDs'].str.split(';') 
    df_peps = df_peps.explode('Evidence ID', ignore_index=True)
    df_peps['Evidence ID']  = df_peps['Evidence ID'].astype(int)

    df = pd.merge(df, df_peps, on=['Evidence ID', 'Sequence', 'Reverse'], how='inner', suffixes=('', '_peps'))
    
    percolator_df = pd.DataFrame()
    percolator_df['SpecId'] = df.apply(lambda x: f"{'decoy' if pd.notna(x['Reverse']) else 'target'}_0_{x['Scan number']}_{x['Raw file']}_{x['Charge']}_1", axis=1)
    percolator_df['Label'] = df['Reverse'].apply(lambda x: -1 if pd.notna(x) else 1) 
    percolator_df['ScanNr'] = df['Scan number']
    percolator_df['ExpMass'] = df['m/z']
    percolator_df['CalcMass'] = df['Mass']
    df['Rank'] = df['Score'].rank(method='dense', ascending=False)
    percolator_df['lnrSp'] = np.log(df['Rank']) 
    df['Last score'] = df['All scores'].str.split(';').str[-1].astype(float)
    percolator_df['deltLCn'] = (df['Score'] - df['Last score'] ) / np.maximum(df['Score'], 1)  # divided by this PSM's score or 1, whichever is larger.
    percolator_df['deltCn'] = (df['Delta score'] ) / np.maximum(df['Score'], 1)
    #percolator_df['Xcorr'] = df['Score']  
    percolator_df['Sp'] = df['Score'] 
    percolator_df['IonFrac'] = df['Peak coverage']
    percolator_df['Mass'] = df['Mass']  
    percolator_df['PepLen'] = df['Length']
    percolator_df['ChargeN'] = df['Charge']

    # if you want charge separated into different columns
    #unique_charges = sorted(df['Charge'].dropna().unique())
    #for charge in unique_charges:
    #    col_name = f'Charge{int(charge)}'
    #    percolator_df[col_name] = (df['Charge'] == charge).astype(int)
    
    percolator_df['enzInt'] = df['Missed cleavages']
    
    #percolator_df['dM'] = df['Mass error [Da]'].fillna(0) # not this, has missing values 
    #percolator_df['absdM'] = df['Mass error [Da]'].abs().fillna(0) #, has missing values 

    if mod == 'sumo':
        df['Modified sequence'] = df['Modified sequence'].str.replace(r'st_o_SUMO2/3-DVFQQQTGG|ST_2_SUMO2/3-DVFQQQTGG|SBM_4_o_SUMO2/3-DVFQQQTGG|SBM_5_o_SUMO2/3-DVFQQQTGG|SBM_3_o_SUMO2/3-DVFQQQTGG|SBM_2_o_SUMO2/3-DVFQQQTGG|SBM_8_o_SUMO2/3-DVFQQQTGG', 'su', regex=True)

        df['Modified sequence'] = df['Modified sequence'].str.replace('Acetyl (Protein N-term)', 'ac', regex=False).str.replace('Oxidation (M)', 'ox', regex=False)
    
        # String formatting peptides from () to [] notation
        df['Modified sequence'] = df['Modified sequence'].str.replace('(', '[', regex=False).str.replace(')', ']', regex=False).str.replace('_', '.', regex=False)
        df['has_su'] = df['Modified sequence'].str.endswith('[su].')

        # SUMO specific, change to your enzyme digestion pettern
        percolator_df['enzN'] = (
            ((df['Amino acid before'].isna()) | 
             (df['Amino acid before'] == 'K') | 
             (df['First amino acid'].isin(['D', 'E'])))
            .astype(int)
        )
        
        percolator_df['enzC'] = (
            (((df['Last amino acid'] == 'K') | 
              (df['Amino acid after'].isin(['D', 'E'])) | 
              (df['Amino acid after'].isna())) &  
             (~df['has_su']) |  
             ((df['has_su']) & (df['Last amino acid'] == 'K') & 
              (df['Amino acid after'].isin(['D', 'E']))))
            .astype(int)
        )

    elif mod == 'none':
        # could add here some code to use trypsin digestion pattern 
        None
    
    # Formatting Peptide sequences
    percolator_df['Peptide'] = df['Modified sequence'] 
    percolator_df['Proteins'] = df['Proteins'].apply(lambda x: ','.join(str(x).split(';')) if pd.notna(x) else 'unknown') 
    
    # Save to output file
    percolator_df.to_csv(output_file, sep='\t', index=False)
    print(f'Converted file saved to {output_file}')

    return percolator_df

In [ ]:
per_df = convert_maxquant_to_percolator(input_file, output_file, peptides_table, mod ='sumo')